<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week7/Day2/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Le reclassement sans serveur Pinecone en action

In [ ]:
import os
from pinecone import Pinecone

# =====================================================================
# 4. DÉFINITION DE LA REQUÊTE ET DES DOCUMENTS CONTEXTUELS (MÉLANGE)
# =====================================================================
query = "Tell me about Apple's products"
documents = [
    "Apples are round, sweet pomaceous fruits produced by the apple tree (Malus domestica).",
    "Apple Inc. designs and manufactures premium consumer electronics including the iPhone, iPad, and Mac computers.",
    "Granny Smith and Honeycrisp are popular commercial varieties of cultivated apples harvested in autumn.",
    "The Apple Watch and AirPods are key hardware devices within the company's wearable technology product ecosystem.",
    "The company was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in April 1976."
]

# Initialisation du client (pc)
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

# =====================================================================
# 5. APPEL DU SERVICE DE RÉÉVALUATION (RERANKING)
# =====================================================================
# Configuration de la structure d'entrée dictionnaires (id/text) requise par Pinecone Inference
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3 # Restriction aux 3 documents les plus discriminants (Remplacement To-Do)
)

# =====================================================================
# 6. EXAMEN ET TRAÇAGE DES RÉSULTATS RECLASSÉS (Remplacement To-Do)
# =====================================================================
def show_reranked_results(query, matches):
    print(f"Query: {query}\n" + "-"*50)
    for i, m in enumerate(matches):
        # Affichage chirurgical de la position ordonnée, du score absolu et de la sémantique
        print(f"Rank {i+1} | Score: {m.score:.4f}")
        print(f"Document Text: {m.document.text}\n")

# L'API moderne de Pinecone stocke le tableau de résultats dans l'attribut '.data'
show_reranked_results(query, reranked.data)


In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Configuration matérielle et specifications géographiques
cloud = os.getenv('PINECONE_CLOUD', 'aws') # Fournisseur cloud cloud computing (To-Do)
region = os.getenv('PINECONE_REGION', 'us-east-1') # Région géographique d'hébergement (To-Do)

spec = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index' # Identifiant unique de l'index vectoriel (To-Do)

# Nettoyage automatique de l'index si un homonyme existe déjà en cache
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# =====================================================================
# CRÉATION DE L'INDEX VECTORIEL SANS SERVEUR (SERVERLESS)
# =====================================================================
pc.create_index(
    name=index_name,
    dimension=384,      # Taille géométrique stricte alignée sur l'encodeur (To-Do)
    metric='cosine',    # Distance mathématique pour le calcul de similarité (To-Do)
    spec=spec
)
print(f"✅ L'index vectoriel sans serveur '{index_name}' a été créé avec succès !")


In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # URL brute GitHub officielle fournie par l'énoncé (To-Do)
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    # Chargement fluide au format JSON Lines (jsonl) dans Pandas
    df = pd.read_json(file_path, orient='records', lines=True)

# =====================================================================
# 2. PRÉVISUALISATION TECHNIQUE ET STATISTIQUE DU DATAFRAME
# =====================================================================
# L'attribut '.shape' renvoie un tuple contenant (nombre de lignes, nombre de colonnes)
print(f"Structure de la matrice de données (Data shape) : {df.shape}")
print("\nAperçu des premières lignes du registre médical :")
display(df.head(3))


In [ ]:
# =====================================================================
# Partie 4 : Génération des embeddings et Upsert dans Pinecone
# =====================================================================
print("--- Partie 4 : Préparation des vecteurs et insertion ---")

# Chargement d'un modèle d'embedding léger (384 dimensions) compatible avec l'index
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Connexion à l'index précédemment créé
index = pc.Index(index_name)

# Préparation des données pour l'insertion par lots (Upsert)
upsert_data = []
for idx, row in df.iterrows():
    # Génération du vecteur numérique pour la note médicale
    vector = embedding_model.encode(row['note']).tolist()

    # Construction du dictionnaire au format strict attendu par Pinecone
    upsert_data.append({
        "id": str(row.get('id', idx)),  # Identifiant unique sous forme de chaîne
        "values": vector,
        "metadata": {
            "patient_id": str(row.get('patient_id', '')),
            "text": row['note']         # Stockage du texte brut pour le décodage final
        }
    })

# Insertion par lots de 100 vecteurs pour optimiser la bande passante de l'API
batch_size = 100
for i in range(0, len(upsert_data), batch_size):
    chunk = upsert_data[i:i + batch_size]
    index.upsert(vectors=chunk)

# Attendre que l'indexation sans serveur soit propagée sur le cloud
print("Attente de la synchronisation de l'index...")
time.sleep(5)
print(f"✅ Insertion terminée ! Nombre total de vecteurs indexés : {index.describe_index_stats()['total_vector_count']}")

# =====================================================================
# Partie 5 : Requête sémantique et Réorganisation (Reranking)
# =====================================================================
print("\n--- Partie 5 : Interrogation et Reranking de sécurité ---")

medical_query = "Show me notes about patients with high blood pressure or hypertension symptoms"

# 1. Encodage vectoriel de la requête médicale
query_vector = embedding_model.encode(medical_query).tolist()

# 2. Recherche sémantique initiale (Top 10) dans Pinecone
search_results = index.query(
    vector=query_vector,
    top_k=10,
    include_metadata=True
)

# Extraction des documents récupérés pour la phase de réorganisation
retrieved_docs = [match['metadata']['text'] for match in search_results['matches']]

# 3. Application du modèle de Reranking pour isoler les notes cliniques les plus cruciales
final_reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=medical_query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(retrieved_docs)],
    top_n=3  # Extraction des 3 meilleures notes d'aide au diagnostic pour le clinicien
)

# 4. Affichage final ordonné et sécurisé pour l'équipe médicale
print("\n" + "="*60)
print("👉 RÉSULTATS CLINIQUES PRIORITAIRES (POST-RERANKING) :")
print("="*60)
for i, item in enumerate(final_reranked.data):
    print(f"Rang {i+1} | Score de pertinence clinique : {item.score:.4f}")
    print(f"Note médicale : {item.document.text}\n")
print("="*60)
